In [1]:
import pandas as pd
import re
from pprint import pprint
import numpy as np
cgg_path = r'c:\Users\glj523\Downloads\LAST_VERSION_OF_CGG3_20240410.xlsx'
cgg_df_orig = pd.read_excel(cgg_path, sheet_name='Sediment_Water', dtype=str)

Aggregate all values in the unamed columns to a single column where values from different columns are seperated by |

In [2]:
cgg_df = cgg_df_orig.copy()
cgg_df = cgg_df.map(lambda x: x.strip() if isinstance(x, str) else x)
cgg_df = cgg_df.replace(['', 'nan'], np.nan)
cgg_df = cgg_df.dropna(how='all', axis='columns')
unnamed_cols = cgg_df.columns[cgg_df.columns.str.startswith('Unnamed')]
cgg_df['from_unnamed_cols'] = cgg_df[unnamed_cols].agg(lambda x: ' | '.join(x.dropna().astype(str)), axis=1).str.strip()
cgg_df = cgg_df.replace('', np.nan)
cgg_df = cgg_df.drop(columns=unnamed_cols)

# Cleaning latitude

Standard cleaning


In [3]:
cgg_df['cleaned_lat'] = (cgg_df.Lat
                        .map(lambda x: re.sub(r'\s+', ' ', x), na_action='ignore')  # Removes any consecutive whitespaces
                        .map(lambda x: x.replace(',', '.'), na_action='ignore') # Standardize decimal point
                        .map(lambda x: x.replace(u'\xa0', u' '), na_action='ignore')  # Fix  weird unicode error
                        )  

Identify unique formats

In [4]:
lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats.unique().tolist()

['+12.12',
 nan,
 '+12',
 '12',
 '12.12',
 '12 12.12 N',
 '12°12′N',
 'N12',
 '12 12 12.12 N',
 "W12 12' 12.12''",
 '12.12 N',
 '12deg12\'12.12"S',
 "N 12º12.12'",
 'N12.12',
 '12.12N',
 "12°12'N",
 "12°12'12''N",
 '12.12°',
 "12°12'12''",
 '12°12\'12"',
 '-12.12',
 'N 12.12',
 '12.12°N',
 '12.12˚N',
 '12˚12\'12.12" N',
 "12 12' 12'' N",
 'N12° 12\' 12.12"',
 '12.12° N',
 '12° 12.12',
 '12°12\'12.12"N',
 '12°12\'12"N',
 '12o12.12',
 '12° 12\' 12.12" N',
 '12°12’12”',
 '12°12\'12.12"',
 "12° 12' 12.12'' N",
 "12° 12'12.12'' N",
 '12° 12.12´S',
 '12° 12’ 12.12” N',
 '12°12’12.12’’N',
 '12° 12.12 N',
 '12°12\'12.12"S',
 '12°12‘12‘‘ N',
 "12' 12.12''",
 '12° 12’ 12.12”',
 '12°12\'12" N',
 '12° 12´12´´N',
 '12°12.12’N',
 '12° 12′ 12″ N']

### Convert to standard characters and symbols
The bad characters were found from the above step

In [5]:
cgg_df['cleaned_lat'] = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r"''|”|’’|‘‘|´´|″", '"', x), na_action='ignore')  # Replaces bad second chars with "
 .map(lambda x: re.sub(r"′|’|´|‘", "'", x), na_action='ignore')  # Replaces bad minute chars with '
 .map(lambda x: re.sub(r"(deg)|º|˚|o", "°", x), na_action='ignore')  # Replaces bad degree chars with °
 )

Put direction indicators (N, S, -, +) in a seperate column

In [6]:
pd.set_option('future.no_silent_downcasting', True)

cgg_df['lat_direction'] = (cgg_df.cleaned_lat
                           .map(lambda x: re.sub(r"[^A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                           .replace(np.nan, '')  # This will make it easier later, when adding the direction back to the coordinate
                           )

#  Removes the direction from the cleaned_lat column, as now it is no longer needed
cgg_df.cleaned_lat = (cgg_df.cleaned_lat
                      .map(lambda x: re.sub(r"[A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                      .str.strip())

Identify unique formats again

In [7]:
from pprint import pprint
lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats = pd.Series(lat_formats.dropna().unique())
print('With degree symbol')
pprint(lat_formats.dropna()[lat_formats.dropna().str.contains('°')].unique().tolist())
print('')
print('without degree symbol')
pprint(lat_formats.dropna()[~lat_formats.dropna().str.contains('°')].unique().tolist())

With degree symbol
["12°12'",
 '12°12\'12.12"',
 "12°12.12'",
 '12°12\'12"',
 '12.12°',
 '12° 12\' 12.12"',
 '12° 12.12',
 '12°12.12',
 '12° 12\'12.12"',
 "12° 12.12'",
 '12° 12\'12"',
 '12° 12\' 12"']

without degree symbol
['12.12',
 '12',
 '12 12.12',
 '12 12 12.12',
 '12 12\' 12.12"',
 '12 12\' 12"',
 '12\' 12.12"']


From the above it seems safe to assume that if
1. A coordinate contains 1 number with or without traling ° that the format is decimal degrees.
2. A coordinate contains 2 numbers seperated with with either a whitespace or ° or both that its formatted as degrees and minutes.
3. A coordinate contains 3 numbers, where first seperator is by ° or whitespace or both and second seperator is ' or whitespace or both, it is formatted as degree minute seconds.

We can formulate this using the following regular expressions:

In [8]:
dd_regex = r'''^\d{1,2}(\.\d+)?(°| °)?$'''
dm_regex = r'''^\d{1,2}( |° |°)\d{1,2}(\.\d+)?('| ')?$'''
dms_regex = r'''^\d{1,2}( |° |°)\d{1,2}( |' |')\d{1,2}(\.\d+)?("| ")?$'''

Checking the general formats 

In [9]:
def check_format(s: str):
    if re.match(dd_regex, s):
        return 'DD'
    elif re.match(dm_regex, s):
        return 'DM'
    elif re.match(dms_regex, s):
        return 'DMS'
    else:
        return 'invalid format'

classified_formats = lat_formats.apply(check_format)

lat_with_formats = pd.DataFrame({'Lat': lat_formats, 'Format': classified_formats}).sort_values(by='Format')
lat_with_formats                           

,Lat,Format
0,12.12,DD
9,12.12°,DD
1,12,DD
2,12 12.12,DM
3,12°12',DM
15,12° 12.12',DM
13,12°12.12,DM
7,12°12.12',DM
12,12° 12.12,DM
14,"12° 12'12.12""",DMS


Apply the same format check to the actual data

In [10]:
cgg_df['lat_format'] = cgg_df.cleaned_lat.map(check_format, na_action='ignore')

Manual check that the lat format identification was done correctly

In [11]:
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    print(lat_formats)
    print()

DD
0     dd.dd
1    dd.dd°
2      d.dd
dtype: object

nan
Series([], dtype: object)

invalid format
0          dddddddd
1            dddddd
2    ddd dd' dd.dd"
3            ddd.dd
4             ddddd
5        dd°dd'ddd"
6        dd' dd.dd"
dtype: object

DM
0      dd dd.dd
1        dd°dd'
2     dd°dd.dd'
3     dd° dd.dd
4      dd°dd.dd
5    dd° dd.dd'
6      d°dd.dd'
dtype: object

DMS
0        dd dd dd.dd
1       dd°dd'dd.dd"
2          dd°dd'dd"
3           dd°d'dd"
4        dd°d'dd.dd"
5         dd dd' dd"
6     dd° dd' dd.dd"
7        dd°dd'd.dd"
8      dd° d' dd.dd"
9      dd° dd'dd.dd"
10       d°dd'dd.dd"
11          dd°dd'd"
12         dd° d'dd"
13       dd° dd' dd"
dtype: object



### Parse

Remove trailing non-numbers to make parsing easier

In [12]:
cgg_df.cleaned_lat = cgg_df.cleaned_lat.map(lambda x: re.sub(r"\D+$", '', x), na_action='ignore')  # removes traling non numbers

##### Split lat data into degree minute and decimal second

If it looks good apply the splitting to the data

In [13]:
cgg_df['cleaned_lat_split'] = cgg_df.cleaned_lat.str.split(pat=r"[ °']+", regex=True)

Manually inspect the splitting

In [14]:
cgg_df['cleaned_lat_split']

0          [61.1399]
1          [61.1399]
2          [61.1399]
3          [61.1399]
4          [61.1399]
            ...     
20852    [64, 33.10]
20853    [64, 33.10]
20854    [64, 33.10]
20855    [64, 33.10]
20856    [64, 33.10]
Name: cleaned_lat_split, Length: 20857, dtype: object

In [15]:
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    
    dms_formats_split = lat_formats.str.split(pat=r"[ °']+", regex=True)

    for raw, splits in zip(lat_formats, dms_formats_split):
        print(raw)
        print(splits)
        
        if ele == 'DD':
            assert len(splits) == 1
        if ele == 'DM':
            assert len(splits) == 2
        if ele == 'DMS':
            assert len(splits) == 3
    print()

DD
dd.dd
['dd.dd']
d.dd
['d.dd']

nan

invalid format
dddddddd
['dddddddd']
dddddd
['dddddd']
ddd dd' dd.dd
['ddd', 'dd', 'dd.dd']
ddd.dd
['ddd.dd']
ddddd
['ddddd']
dd°dd'ddd
['dd', 'dd', 'ddd']
dd' dd.dd
['dd', 'dd.dd']

DM
dd dd.dd
['dd', 'dd.dd']
dd°dd
['dd', 'dd']
dd°dd.dd
['dd', 'dd.dd']
dd° dd.dd
['dd', 'dd.dd']
d°dd.dd
['d', 'dd.dd']

DMS
dd dd dd.dd
['dd', 'dd', 'dd.dd']
dd°dd'dd.dd
['dd', 'dd', 'dd.dd']
dd°dd'dd
['dd', 'dd', 'dd']
dd°d'dd
['dd', 'd', 'dd']
dd°d'dd.dd
['dd', 'd', 'dd.dd']
dd dd' dd
['dd', 'dd', 'dd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
dd°dd'd.dd
['dd', 'dd', 'd.dd']
dd° d' dd.dd
['dd', 'd', 'dd.dd']
dd° dd'dd.dd
['dd', 'dd', 'dd.dd']
d°dd'dd.dd
['d', 'dd', 'dd.dd']
dd°dd'd
['dd', 'dd', 'd']
dd° d'dd
['dd', 'd', 'dd']
dd° dd' dd
['dd', 'dd', 'dd']



Test if the different formats have correct number of elementes in the split lists and that each element only contains numbers

In [16]:
def all_numbers(lst):
    return all(s.replace(".", "", 1).isdigit() for s in lst)

for i, row in cgg_df[['lat_format', 'cleaned_lat_split']].iterrows():
    split_lst = row.cleaned_lat_split
    lat_format = row.lat_format
    
    if lat_format == 'DD':
        assert len(split_lst) == 1
    if lat_format == 'DM':
        assert len(split_lst) == 2
    if lat_format == 'DMS':
        assert len(split_lst) == 3
    if isinstance(split_lst, list):
        for ele in split_lst:
            try:
                float(ele)
            except ValueError:
                raise Exception(f'bad list: {split_lst}')

Convert direction to plus and minus

In [17]:
def convert_direction(row):
    direction = str(row.lat_direction)
    if direction == 'nan':
        return np.nan
    direction = re.sub(r'[Nn]', '+', direction)
    direction = re.sub(r'[Ss]', '-', direction)
    direction = direction.strip()
    if bool(re.match(r'^(\-|\+)$', direction)) or direction == '':
        return direction
    else:
        return 'invalid direction'

cgg_df['converted_lat_direction'] = cgg_df.apply(convert_direction, axis=1)

Convert Split DM data into decimal degrees

In [18]:
def add_direction(row):
    direction = str(row.converted_lat_direction)
    coord = row.converted_lat
    if not direction == 'invalid direction':
        return float(str(direction) + str(coord))
    else:
        return coord

def convert_dd(lst):
    assert len(lst) == 1
    return float(lst[0])

def convert_dm(lst):
    assert len(lst) == 2
    degrees, minutes = float(lst[0]), float(lst[1])
    
    return degrees + (minutes/60)

def convert_dms(lst):
    assert len(lst) == 3
    degrees, minutes, seconds = float(lst[0]), float(lst[1]), float(lst[2])
    
    return degrees + (minutes/60) + (seconds/3600)

def convert_to_dd(row):
    lat_format = row.lat_format 
    split_lst = row.cleaned_lat_split
    if lat_format == 'DD':
        return convert_dd(split_lst)
    elif lat_format == 'DM':
        return convert_dm(split_lst)
    elif lat_format == 'DMS':
        return convert_dms(split_lst)
    else:
        return np.nan
    
cgg_df['converted_lat'] = cgg_df.apply(convert_to_dd, axis=1)
cgg_df['converted_lat'] = cgg_df.apply(add_direction, axis=1)

Check that the conversion was done correctly

In [19]:
lat_formats = (cgg_df.converted_lat
 .map(lambda x: re.sub(r'\d+', '12', str(x)), na_action='ignore')  # Turns all n sized numbers into 123
 )
lat_formats.unique()

array(['12.12', nan, '-12.12'], dtype=object)

Manually inspect the data to verify

In [20]:
cgg_df.query('lat_format == "DMS"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
17584,64° 8´22´´N,64° 8'22,64.139444,DMS,N
17513,"64°51'0"" N",64°51'0,64.850000,DMS,N
7515,"69˚3'29.35"" N",69°3'29.35,69.058153,DMS,N
7544,"69˚3'29.35"" N",69°3'29.35,69.058153,DMS,N
15511,"55°40'44.352""",55°40'44.352,55.678987,DMS,
10456,"34°35'2.19""N",34°35'2.19,34.583942,DMS,N
10397,"34°35'2.19""N",34°35'2.19,34.583942,DMS,N
2237,"74 53 49,66 N",74 53 49.66,74.897128,DMS,N
15671,65°47‘52‘‘ N,65°47'52,65.797778,DMS,N
7553,"69˚3'29.35"" N",69°3'29.35,69.058153,DMS,N


In [21]:
cgg_df.query('lat_format == "DM"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
3652,N 73º09.387',73°09.387,73.156450,DM,N
5072,52°04'N,52°04,52.066667,DM,N
5075,52°04'N,52°04,52.066667,DM,N
5071,52°04'N,52°04,52.066667,DM,N
5034,52°04'N,52°04,52.066667,DM,N
10386,50° 32.932,50° 32.932,50.548867,DM,
10325,50° 32.932,50° 32.932,50.548867,DM,
5120,52°04'N,52°04,52.066667,DM,N
3647,N 73º21.025',73°21.025,73.350417,DM,N
5168,52°04'N,52°04,52.066667,DM,N


In [22]:
cgg_df.query('lat_format == "DD"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
14009,30.512615,30.512615,30.512615,DD,
14185,30.811696,30.811696,30.811696,DD,
11597,65.7085˚N,65.7085,65.708500,DD,N
3860,N57.69979,57.69979,57.699790,DD,N
5783,55.685982°,55.685982,55.685982,DD,
18725,55.533435,55.533435,55.533435,DD,
11403,66.0610˚N,66.0610,66.061000,DD,N
19730,"60,395964",60.395964,60.395964,DD,
14988,29.88603,29.88603,29.886030,DD,
13002,7.012500 N,7.012500,7.012500,DD,N


In [23]:
cgg_df.query('lat_format == "invalid format"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
20817,64' 33.10'',64' 33.10,NaN,invalid format,
20764,64' 33.10'',64' 33.10,NaN,invalid format,
3117,W116 11' 47.538'',116 11' 47.538,NaN,invalid format,W
20625,61' 57.50'',61' 57.50,NaN,invalid format,
3153,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
16894,61' 56.88'',61' 56.88,NaN,invalid format,
2517,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
16327,64' 33.10'',64' 33.10,NaN,invalid format,
3144,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
2903,W119 34' 59.88'',119 34' 59.88,NaN,invalid format,W


In [24]:
cgg_df.to_csv('cgg_clean_lat.tsv', sep='\t')

### Clean invalid formatted coordinates

Check if any lat directions are bad

Mark rows that contain E, W, e or w in lat as invalid, bad_direction

In [52]:
invalid_filter = cgg_df.lat_direction.str.contains(r'E|W|e|w', na=False)

In [53]:
cgg_df[invalid_filter]['invalid_format'] = True
cgg_df[invalid_filter]['invalid_format_comment'] = 'bad_direction'

C:\Users\glj523\AppData\Local\Temp\ipykernel_3732\311128983.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cgg_df[invalid_filter]['invalid_format'] = True
C:\Users\glj523\AppData\Local\Temp\ipykernel_3732\311128983.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cgg_df[invalid_filter]['invalid_format_comment'] = 'bad_direction'


Do format checking on invalid ones, replace digits with d

Lats that are probably missing a decimal

In [ ]:
no_dec_filter = cgg_df.cleaned_lat.map(lambda x: bool(re.match(pattern=r'^\d+$', string=x)), na_action='ignore')
sorted(cgg_df[no_dec_filter].cleaned_lat.unique())

ValueError: Cannot mask with non-boolean array containing NA / NaN values